[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/255ribeiro/cadquery_basics/blob/master/docs/tuto_colab_build/build123d_basic_gc.ipynb)

# Introdução e Conceitos Básicos
## build123d para Arquitetos e Engenheiros
### Versão Google Colab

---

### O que é o build123d?

**build123d** é uma biblioteca Python de modelagem geométrica 3D baseada em código — a mesma proposta do CadQuery, e construída sobre o mesmo núcleo geométrico **OCCT (Open Cascade Technology)**, o motor usado pelo FreeCAD. Ele trabalha com geometria sólida real (B-Rep), o que significa que os modelos gerados têm volume, massa e propriedades físicas precisas.

Essa abordagem traz vantagens importantes para arquitetos e projetistas:

- **Parametrização total**: qualquer dimensão pode ser uma variável, facilitando variações do projeto
- **Reprodutibilidade**: o código é a documentação do modelo
- **Automação**: é possível gerar dezenas de variações automaticamente
- **Integração**: o modelo pode ser conectado a planilhas, bancos de dados e outras ferramentas

O build123d oferece dois estilos de escrita: o **modo Builder** (com blocos `with BuildPart() as ...:`, mais próximo de um histórico de operações de um software CAD) e o **modo Álgebra**, que cria e combina objetos diretamente com operadores (`+`, `-`, `&`). Neste curso usamos principalmente o **modo Álgebra**, por ser o mais direto para quem já pensa em termos de volumes — `Box(...)`, `Cylinder(...)`, `Sphere(...)` — sem precisar de um `Workplane` para começar.

---

### Visualização no Google Colab

Nesta versão do curso utilizamos o **Google Colab** como ambiente de execução. Como não existe (ainda) uma biblioteca de visualização publicada para build123d equivalente à `cadquery-simpleviewer`, este curso usa um pequeno módulo próprio — **`build123d_simpleviewer`** — que faz exatamente a mesma coisa (renderiza os modelos com Plotly, direto na célula do notebook) mas adaptado aos tipos de objeto do build123d. O módulo é baixado automaticamente do repositório do curso na célula de instalação abaixo.

> ⚠️ **Antes de começar**: execute a célula de instalação abaixo. Ela precisa rodar apenas uma vez por sessão. Se o ambiente do Colab for reiniciado, execute-a novamente.

---

## Instalação

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print("Running in Colab, installing packages...")
    !pip install build123d
    !wget -q -N https://raw.githubusercontent.com/255ribeiro/cadquery_basics/master/docs/tuto_colab_build/build123d_simpleviewer.py
else:
    print("Not running in Colab, skipping package installation.")

## Importações

In [ ]:
from build123d import *
from build123d_simpleviewer import show

---

## Primeiro Exemplo

Antes de entrar nos detalhes da biblioteca, veja como é simples criar e visualizar um objeto 3D com o build123d. O código abaixo cria uma caixa e a exibe com a função `show()`:

In [ ]:
caixa = Box(5, 3, 2)

show(caixa)

Em apenas duas linhas: uma cria a geometria, a outra a exibe. O gráfico gerado é interativo — você pode **orbitar**, **aproximar** e **afastar** com o mouse diretamente na célula.

Nos próximos tópicos vamos entender o que cada parte desse código significa.

---

## Objetos e Planos

Diferente do CadQuery, o build123d não exige um `Workplane` para criar uma primitiva — `Box(5, 3, 2)` já é, por si só, um sólido posicionado na origem `(0, 0, 0)`. O conceito de **plano de trabalho** continua existindo (é assim que se define sobre qual superfície uma operação 2D acontece, ou para onde um sólido é deslocado), mas ele aparece como a classe **`Plane`**, usada quando você precisa dela — não como ponto de partida obrigatório.

Os três planos principais já vêm prontos como constantes:

| Nome | Plano | Uso comum |
|------|-------|-----------|
| `Plane.XY` | Horizontal (chão) | Plantas baixas, lajes |
| `Plane.XZ` | Frontal (fachada) | Elevações frontais |
| `Plane.YZ` | Lateral | Cortes laterais |

---

### Definindo uma origem personalizada

Para posicionar um objeto a partir de um ponto específico (e não da origem `(0, 0, 0)`), multiplicamos um `Plane` deslocado pelo objeto:

```python
Plane(origin=(x, y, z)) * Box(l, w, h)
```

O operador `*` aplica a localização do plano ao objeto à direita — o resultado é uma cópia do objeto posicionada naquele ponto. É o equivalente direto ao `cq.Workplane("XY", origin=(x, y, z))` do CadQuery.

Veja a comparação: quatro caixas idênticas criadas com origens diferentes.

In [ ]:
# Caixa na origem padrão (0, 0, 0)
caixa_centro = Box(60, 60, 60)

# Caixa com origem deslocada 120mm no eixo X
caixa_direita = Plane(origin=(120, 0, 0)) * Box(60, 60, 60)

# Caixa com origem deslocada 120mm no eixo Y
caixa_frente = Plane(origin=(0, 120, 0)) * Box(60, 60, 60)

# Caixa com origem elevada 60mm no eixo Z
# Útil para empilhar volumes, como pavimentos de um edifício
caixa_cima = Plane(origin=(0, 0, 60)) * Box(60, 60, 60)

show(
    [caixa_centro, caixa_direita, caixa_frente, caixa_cima],
    names=["Origem (0,0,0)", "X deslocado", "Y deslocado", "Z elevado"]
)

> 💡 **Dica**: usar `Plane(origin=...)` é especialmente útil quando você quer empilhar volumes (como lajes e pavimentos) ou alinhar elementos a uma grade de projeto. Para deslocamentos simples, `Pos(x, y, z) * objeto` funciona da mesma forma e é um pouco mais direto de ler.

---

## Formas Primitivas Básicas

O build123d oferece formas geométricas prontas chamadas **primitivas**. São o ponto de partida para construções mais complexas. Todas são **centradas na origem por padrão** (controlado pelo parâmetro `align`, que veremos no próximo notebook).

| Classe | Descrição | Parâmetros principais |
|--------|-----------|----------------------|
| `Box(l, w, h)` | Caixa retangular | comprimento, largura, altura |
| `Cylinder(r, h)` | Cilindro | raio, altura |
| `Sphere(r)` | Esfera | raio |
| `Cone(r1, r2, h)` | Cone/Tronco de cone | raio da base, raio do topo, altura |

> ⚠️ **Atenção à ordem dos parâmetros**: no CadQuery, `.cylinder(height, radius)` recebe a altura primeiro. No build123d, `Cylinder(radius, height)` recebe o **raio primeiro**. Use argumentos nomeados (`Cylinder(radius=25, height=120)`) para evitar confusão — é o estilo adotado neste curso.

---

## Visualização: a função `show()`

O módulo **build123d_simpleviewer** fornece a função `show()` para exibir objetos build123d como gráficos 3D interativos diretamente na célula — com a mesma assinatura da versão CadQuery deste curso.

A função aceita uma **mistura livre** de tipos na mesma chamada:

| Tipo de objeto | Como é exibido |
|----------------|----------------|
| Sólido (`Part`, `Solid`, `Compound`, `Sketch`, `Face`) | Malha 3D sólida tessellada |
| `Edge` / `Wire` | Linha amostrada ao longo da curva |
| `Vector` / `Vertex` / `[x, y, z]` | Marcador pontual |

### Sintaxe básica

```python
# Um único objeto
show(objeto)

# Vários objetos
show([objeto_a, objeto_b], names=["Nome A", "Nome B"])
```

### Parâmetros principais

| Parâmetro | Padrão | Descrição |
|-----------|--------|-----------|
| `objects` | — | Objeto ou lista — qualquer mix de sólidos, arestas, wires e pontos |
| `names` | `None` | Nomes para a legenda |
| `colors` | `None` | Cor de cada sólido. Consulte [Plotly CSS Colors](https://plotly.com/python/css-colors/) |
| `opacity` | `1.0` | Transparência dos sólidos — `1.0` = opaco, `0.0` = invisível |
| `visible_axes` | `"xyz"` | Eixos visíveis. `None` oculta todos. Combinações: `"x"`, `"y"`, `"z"`, `"xy"`, `"xz"`, `"yz"`, `"xyz"` |
| `z` | `None` | Cota do plano de chão. `None` = sem plano |
| `plane_color` | `"whitesmoke"` | Cor do plano de chão |
| `plane_size` | `50` | Metade do lado do plano de chão |
| `plane_opacity` | `0.8` | Transparência do plano de chão |
| `tessellation_tolerance` | `0.01` | Precisão do mesh — menor = mais fino, mais lento |
| `padding` | `0.15` | Margem adicionada ao redor da geometria |
| `points_display` | `None` | Estilo dos marcadores de ponto. Chaves: `size`, `color`, `symbol` (`"circle"`, `"square"`, `"diamond"`, `"cross"`, `"x"`), `opacity` |
| `lines_display` | `None` | Estilo das linhas (arestas e wires). Chaves: `color`, `width`, `mode` (`"lines"` ou `"lines+markers"`), `samples` (pontos amostrados por aresta, padrão `50`), `opacity` |

### Controles interativos do visualizador

Após executar `show()`, o gráfico exibe botões no topo:

- **X ● / X ○**, **Y ● / Y ○**, **Z ● / Z ○** — liga e desliga cada eixo individualmente
- **Camera** — alterna entre projeção Perspectiva e Ortográfica

Com o mouse: arraste para orbitar, scroll para zoom, arraste com botão direito para pan.

---

Vamos ver os diferentes modos de uso. Primeiro criamos os objetos:

In [ ]:
# Caixa com 80 x 80mm de base e 40mm de altura
caixa = Box(80, 80, 40)

# Cilindro com raio 25mm e altura 120mm
cilindro = Cylinder(radius=25, height=120)

### Modo técnico — eixos visíveis (padrão)

In [ ]:
show([caixa, cilindro], names=["Caixa", "Cilindro"])

### Modo limpo — sem eixos, com plano de chão

In [ ]:
show(
    [caixa, cilindro],
    names=["Caixa", "Cilindro"],
    visible_axes=None,
    z=0
)

### Com cores e plano personalizado

In [ ]:
show(
    [caixa, cilindro],
    names=["Caixa", "Cilindro"],
    colors=["lightgray", "steelblue"],
    visible_axes=None,
    z=0,
    plane_color="gainsboro",
    plane_size=200
)

> 🔎 **Resumo prático**:
> - Use `visible_axes="xyz"` (padrão) durante a modelagem para verificar dimensões e posições
> - Use `visible_axes=None` com `z=` para apresentações com visual limpo

---

## Usando Variáveis para Parametrizar

Uma das grandes vantagens de modelar em código é usar **variáveis** para as dimensões. Mudar o projeto inteiro se torna tão simples quanto alterar um único número.

In [ ]:
# Dimensões da caixa — experimente mudar e executar novamente!
caixa_comprimento = 80    # mm
caixa_largura     = 80    # mm
caixa_altura      = 40    # mm

# Dimensões do cilindro
cilindro_raio   = 25      # mm
cilindro_altura = 120     # mm

# Criando os objetos com as variáveis
caixa_param    = Box(caixa_comprimento, caixa_largura, caixa_altura)
cilindro_param = Cylinder(radius=cilindro_raio, height=cilindro_altura)

show([caixa_param, cilindro_param], names=["Caixa", "Cilindro"])

---

## Movendo Objetos: `.translate()`

Para posicionar um sólido no espaço após criá-lo, usamos o método `.translate()` — com a mesma sintaxe do CadQuery. Ele recebe uma **tupla** com três valores: `(x, y, z)`.

- `x` → move para a direita (positivo) ou esquerda (negativo)
- `y` → move para frente (positivo) ou trás (negativo)
- `z` → move para cima (positivo) ou baixo (negativo)

> 💡 **`Plane(origin=...)` ou `.translate()`?** Multiplicar por um `Plane` deslocado define o ponto de partida *antes* de criar a forma (ou reposiciona um objeto existente do zero). O `.translate()` move a forma *a partir de sua posição atual*. Para posicionamento simples os dois chegam ao mesmo resultado. Use o que tornar o código mais fácil de ler.

In [ ]:
# Caixa posicionada na origem
caixa_fixa = Box(80, 80, 40)

# Cilindro criado na origem e depois deslocado 120mm para a direita
cilindro_mov = Cylinder(radius=25, height=120)
cilindro_mov = cilindro_mov.translate((120, 0, 0))

show(
    [caixa_fixa, cilindro_mov],
    names=["Caixa", "Cilindro deslocado"],
    visible_axes=None,
    z=0,
    plane_size=300
)

---

## Rotacionando Objetos: `.rotate()`

Para rotacionar um sólido, usamos `.rotate()`. Diferente do CadQuery — que define o eixo com **dois pontos** — o build123d usa a classe **`Axis`**, que representa um eixo como uma origem mais uma direção. Os três eixos principais já vêm prontos como constantes:

```python
objeto.rotate(eixo, angulo)
```

| Eixo | Constante | Efeito visual |
|------|-----------|---------------|
| Z | `Axis.Z` | Girar em planta |
| X | `Axis.X` | Inclinar para frente/trás |
| Y | `Axis.Y` | Inclinar para os lados |

Para um eixo qualquer (não coincidente com os principais), cria-se um `Axis(origem, direção)` — o equivalente aos dois pontos do CadQuery, mas descrito como ponto de partida + vetor em vez de dois pontos.

In [ ]:
# Placa retangular na posição original
placa_original = Box(120, 20, 10)

# A mesma placa rotacionada 45° no plano horizontal (em torno do eixo Z)
placa_girada = Box(120, 20, 10)
placa_girada = placa_girada.rotate(Axis.Z, 45)

show(
    [placa_original, placa_girada],
    names=["Original", "Rotacionada 45°"]
)

---

## Escalando Objetos

Diferente do CadQuery — que não tem um método de escala acessível diretamente e exige recorrer ao motor OCCT por baixo dos panos — o build123d já traz a operação **`scale()`** pronta para uso, inclusive com **escala independente por eixo**:

```python
scale(objeto, by=(sx, sy, sz))
```

- `by` pode ser um único número (escala uniforme) ou uma tupla `(sx, sy, sz)` (escala independente por eixo)
- Um fator `1.0` mantém a dimensão original. Um fator `2.0` dobra. Um fator `0.5` reduz à metade.

> ⚠️ **Atenção**: a escala é aplicada em relação à **origem** `(0, 0, 0)` (ou ao ponto passado em `about=`). É recomendado escalar *antes* de transladar.

In [ ]:
esfera_original = Sphere(30)

esfera_uniforme = scale(esfera_original, by=2.0)
esfera_escala_x = scale(esfera_original, by=(3.0, 1.0, 1.0))
esfera_escala_y = scale(esfera_original, by=(1.0, 3.0, 1.0))
esfera_escala_z = scale(esfera_original, by=(1.0, 1.0, 3.0))

esfera_uniforme = esfera_uniforme.translate(( 120, 0, 0))
esfera_escala_x = esfera_escala_x.translate(( 240, 0, 0))
esfera_escala_y = esfera_escala_y.translate(( 360, 0, 0))
esfera_escala_z = esfera_escala_z.translate(( 480, 0, 0))

show(
    [esfera_original, esfera_uniforme, esfera_escala_x, esfera_escala_y, esfera_escala_z],
    names=["Original", "Uniforme (x2)", "Escala X (x3)", "Escala Y (x3)", "Escala Z (x3)"]
)

> 🔎 **Observe**: a esfera original permanece esférica. A cópia uniforme também é esférica, porém maior. As três últimas se deformam em elipsoides — cada uma alongada em uma direção diferente.

---

## Exportando o Modelo

O build123d permite exportar os modelos para diferentes formatos através de funções dedicadas (em vez do `cq.exporters.export()` único do CadQuery):

| Formato | Extensão | Função | Uso |
|---------|----------|--------|-----|
| STEP | `.step` | `export_step()` | Troca com outros softwares CAD (Rhino, FreeCAD, Fusion 360) |
| STL  | `.stl`  | `export_stl()`  | Impressão 3D, renderização |
| DXF  | `.dxf`  | `export_dxf()`  | Compatível com AutoCAD |

> 💡 **No Google Colab**: os arquivos são salvos no sistema de arquivos temporário do Colab. Para baixar, clique no ícone de pasta no painel esquerdo, localize o arquivo e clique em "Baixar".

In [ ]:
modelo_exportar = Box(100, 80, 50)

export_step(modelo_exportar, "meu_modelo.step")
export_stl(modelo_exportar, "meu_modelo.stl")

print("Arquivos exportados com sucesso!")
print("Acesse o painel de arquivos do Colab (ícone de pasta à esquerda) para baixá-los.")

---

## Exercício

Crie um modelo simples que represente **uma composição de três volumes arquitetônicos**: uma torre cilíndrica central e dois blocos retangulares nos lados.

**Requisitos:**
1. A torre cilíndrica deve ser visivelmente mais alta que os blocos
2. Use `Plane(origin=...)` para posicionar pelo menos um dos volumes
3. Use `.translate()` para posicionar pelo menos um dos volumes
4. Use variáveis para todas as dimensões
5. Exiba com `show()` usando `visible_axes=None` e `z=0`
6. **Desafio**: use `colors` para atribuir uma cor diferente a cada volume

In [ ]:
# Escreva seu código aqui


---

## Resumo

Neste notebook você aprendeu:

- O que é o build123d e por que é útil para arquitetos
- Como instalar o build123d no Google Colab e carregar o `build123d_simpleviewer`
- Que primitivas como `Box(...)` já são sólidos completos, sem precisar de um `Workplane` — e como usar a classe **`Plane`** para definir uma **origem personalizada** com `Plane(origin=(x, y, z))`
- As formas primitivas: `Box`, `Cylinder`, `Sphere`, `Cone` — e a ordem de parâmetros (raio antes da altura)
- A função **`show()`** e seus parâmetros — incluindo `visible_axes`, `z`, `points_display` e `lines_display`
- Os tipos aceitos por `show()`: sólidos, `Edge`, `Wire`, pontos (`Vector`, `Vertex`, `[x, y, z]`)
- Como usar **variáveis** para parametrizar o modelo
- Como **posicionar** objetos com `.translate()`
- Como **rotacionar** objetos com `.rotate()` e a classe `Axis`
- Como **escalar** objetos — uniforme ou por eixo — com a operação nativa `scale()`
- Como **exportar** modelos para STEP e STL

No próximo notebook veremos como realizar **operações booleanas** — união, subtração e interseção — para criar formas mais complexas a partir da combinação de sólidos simples.

---
*build123d para Arquitetos — Notebook 1 (versão Google Colab)*